In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_validate
from sklearn.model_selection import ShuffleSplit
from sklearn import metrics
from numpy import mean
from numpy import std
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
# Models Training with selective features and training dataset:

In [ ]:
df_1= pd.read_csv('dataset.csv') #Import the inhibitor dataset
df_1 = df_1.dropna()
df_clean = df_1.set_index('compound')
df_clean

In [ ]:
def categorize_ic50(ic50):
    if ic50 <= 1:
        return 'Strong'
    else:
        return 'Weak' #Define the inhibition potency

df_1['Inhibitor_Category'] = df_1['IC50'].apply(categorize_ic50)

In [ ]:
RF_features = df_clean.loc[:,['MaxEStateIndex', 'SMR_VSA10', 'MaxAbsEStateIndex', 'VSA_EState4', 'BCUT2D_MRLOW', 
                              'fr_bicyclic', 'SlogP_VSA9', 'MaxPartialCharge', 'MinAbsPartialCharge', 'VSA_EState10', 
                              'VSA_EState5', 'SlogP_VSA7', 'MolLogP', 'VSA_EState2', 'Kappa1', 'qed', 'fr_NH1', 
                              'FpDensityMorgan3', 'MinAbsEStateIndex', 'NumHAcceptors']]

XGB_features = df_clean.loc[:,['SMR_VSA10', 'VSA_EState5', 'qed', 'MaxEStateIndex', 'Kappa3', 'VSA_EState4', 'SlogP_VSA1', 
                               'fr_bicyclic', 'BCUT2D_MRLOW', 'VSA_EState2', 'MinAbsEStateIndex', 'BCUT2D_LOGPLOW', 'EState_VSA9', 
                               'fr_Ar_OH', 'BCUT2D_CHGHI', 'SlogP_VSA8', 'NumHAcceptors', 'NumHeteroatoms', 
                               'Kappa1', 'MaxPartialCharge']]

x= np.array(RF_features) # Change to XGB_features for the XGBoost Model
scaler = StandardScaler()
scaler.fit(x)
x = scaler.transform(x)

feature_names = [f"Feature {i}" for i in range(x.shape[1])]

In [ ]:
df_1['Inhibitor_Category'].astype('category')
labelencoder = LabelEncoder()
y= labelencoder.fit_transform(df_1['Inhibitor_Category'])
y = np.array(y)
# 0 is potent, 1 is non-potent

In [ ]:
# Random Forest model training with seletive features

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

rf_sel = RandomForestClassifier(random_state=0, n_estimators=1400, min_samples_split = 2, min_samples_leaf=1, max_features= 'auto', max_depth=None,
                           criterion ='entropy', bootstrap= True)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

rf_sel.fit(X_train, y_train)

# Generate predictions
y_pred = rf_sel.predict(X_test)

In [ ]:
# XGBoost training with selective features

In [ ]:
from xgboost import XGBClassifier

# Build XGBoost model with selected hyperparameters
xgb_sel = XGBClassifier(random_state=0, colsample_bytree = 1.0, gamma = 0.2, learning_rate = 0.1, max_depth = 3, 
                    min_child_weight = 1, n_estimators = 100, subsample = 1.0)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

xgb_sel.fit(X_train, y_train)

# Generate predictions
y_pred = xgb_sel.predict(X_test)

In [ ]:
# Predicting external dataset

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load the molecule data from a CSV file
df_val = pd.read_csv('example.csv')

# Save the "compound" column separately for result labelling
molecule_ids = df_val['compound'] if 'compound' in df_val.columns else pd.Series([f'mol_{i}' for i in range(len(df_val))])

RF_features_v = df_val.loc[:,['MaxEStateIndex', 'SMR_VSA10', 'MaxAbsEStateIndex', 'VSA_EState4', 'BCUT2D_MRLOW', 
                              'fr_bicyclic', 'SlogP_VSA9', 'MaxPartialCharge', 'MinAbsPartialCharge', 'VSA_EState10', 
                              'VSA_EState5', 'SlogP_VSA7', 'MolLogP', 'VSA_EState2', 'Kappa1', 'qed', 'fr_NH1', 
                              'FpDensityMorgan3', 'MinAbsEStateIndex', 'NumHAcceptors']]

XGB_features_v = df_val.loc[:,['SMR_VSA10', 'VSA_EState5', 'qed', 'MaxEStateIndex', 'Kappa3', 'VSA_EState4', 'SlogP_VSA1', 
                               'fr_bicyclic', 'BCUT2D_MRLOW', 'VSA_EState2', 'MinAbsEStateIndex', 'BCUT2D_LOGPLOW', 'EState_VSA9', 
                               'fr_Ar_OH', 'BCUT2D_CHGHI', 'SlogP_VSA8', 'NumHAcceptors', 'NumHeteroatoms', 
                               'Kappa1', 'MaxPartialCharge']]

# Drop non-feature columns if needed
x_RF = np.array(RF_features_v)
x_XGB = np.array(XGB_features_v)

scaler = StandardScaler()
scaler.fit(x_RF)
x_RF = scaler.transform(x_RF)
scaler.fit(x_XGB)
x_XGB = scaler.transform(x_XGB)

# Predict using both models
xgb_predictions = xgb_sel.predict(x_XGB)
rft_predictions = rf_sel.predict(x_RF)

# XGBoost confidence
xgb_probs = xgb.predict_proba(x_XGB)[:, 1]  # Probability of being non-potent

# Random Forest confidence
rf_probs = rtf.predict_proba(x_RF)[:, 1]  

# Convert numeric predictions to labels
label_map = {0: 'potent', 1: 'non-potent'}
xgb_labels = [label_map[pred] for pred in xgb_predictions]
rft_labels = [label_map[pred] for pred in rft_predictions]

# Combine results into a DataFrame
results = pd.DataFrame({
    'Molecule': molecule_ids,
    'XGBoost_Prediction': xgb_labels,
    'XGB_Probablity': xgb_probs,
    'RandomForest_Prediction': rft_labels,
    'RF_Probablity': rf_probs,
})

# Save results to a new CSV or Excel file
results.to_csv('molecule_predictions.csv', index=False)

print("Predictions saved to 'molecule_predictions.csv'")
